# Bartons indicator expressions

Bartons indicators return native Polars expressions. This tutorial uses the bundled daily sample prices to demonstrate selection, composition, multi-output structs, and lazy queries.

In [ ]:
import polars as pl

from bartons.indicators import ATR, DMI, EMA, MACD, MOM, RSI, SMA, WILLR
from bartons.samples import sample_prices

## Load the sample prices

`sample_prices` ships with Bartons, so the notebook requires no network access.

In [ ]:
prices = sample_prices("daily", max_bars=250)
prices.head()

## Select indicators

Factories such as `EMA(20)` and `RSI(14)` produce expressions. Their default output names are the lowercase indicator names; use `alias` when selecting the same indicator more than once.

In [ ]:
signals = prices.select(
    "date",
    "close",
    EMA(20),
    SMA(20),
    RSI(14),
    ATR(14),
    MOM(10),
    WILLR(14),
)
signals.tail()

## Compose expressions

Single-source indicators default to `close`. A source can also be supplied explicitly, passed as the first argument, or piped from another expression.

In [ ]:
composed = prices.select(
    "date",
    EMA(10).alias("ema_close"),
    EMA(10, src=pl.col("high")).alias("ema_high"),
    pl.col("close").pipe(EMA, 10).pipe(RSI, 14).alias("rsi_of_ema"),
)
composed.tail()

## Multi-output indicators

Multi-output indicators are native struct expressions. Keep the structs as columns during the query, then unnest the result when top-level fields are useful.

In [ ]:
multi = prices.select("date", "close", MACD(), DMI())
multi.tail()

In [ ]:
multi.unnest("macd", "dmi").tail()

## Lazy queries

The same indicator expressions work in lazy pipelines and can be referenced by later query stages.

In [ ]:
overbought = (
    prices.lazy()
    .with_columns(RSI(14), EMA(20))
    .filter(pl.col("rsi") > 70)
    .select("date", "close", "ema", "rsi")
    .collect()
)
overbought.tail()